In [ ]:
"""
exp165: CatBoost Stage5 本格HP最適化 — Kaggle GPU (CUDA) 自己完結版

CatBoostのGPUはNVIDIA CUDA専用（Apple Silicon Metal/MPS非対応、ローカルで確認済み）。
CatBoostの確定特徴量セット（bin化, H-CB-001でLB検証済み・生値より優位）を使い、
Optunaで本格HP探索（時間予算: 5時間、GPUクォータ6時間/週の範囲内）を行う。

train_features_slim.pkl / test_features_slim.pkl は Dataset kakiginobuya/s6e7-features-slim
から読み込む（bin5/bin7特徴量、RealMLP Stage1.5用に作成したものを再利用）。

出力: /kaggle/working/best_params_cb_full.json, oof_165_cb_full.npy, test_165_cb_full.npy
"""

import json
import time
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Dataset パス自動検出 ──────────────────────
DATASET_NAME = "s6e7-features-slim"
_KAGGLE_INPUT = Path("/kaggle/input")
_ds_candidates = [
    _KAGGLE_INPUT / "datasets" / "kakiginobuya" / DATASET_NAME,
    _KAGGLE_INPUT / DATASET_NAME,
]
DATASET_DIR = next((p for p in _ds_candidates if p.exists()), _ds_candidates[0])
print(f"DATASET_DIR = {DATASET_DIR}")

TARGET_COL = "health_condition"
N_CLASSES = 3
N_SPLITS = 5
SEED = 42
TIME_BUDGET_SECONDS = 5 * 3600  # 5時間（GPUクォータ6時間/週に対し余裕を残す）

FEATURES = [
    "sleep_duration_raw_nan", "heart_rate_bin5", "bmi_bin5", "calorie_expenditure_bin7",
    "step_count_bin7", "exercise_duration_bin5", "water_intake_bin5",
    "gender", "physical_activity_level", "sleep_quality", "smoking_alcohol", "stress_level", "diet_type",
]

BETA_GRID = [0.0, 0.25, 0.5, 0.75, 1.0, 1.15, 1.3, 1.5, 1.75, 2.0, 2.5]


def calibrated_score(oof: np.ndarray, y_raw: pd.Series, classes: np.ndarray, prior: np.ndarray) -> float:
    return max(
        balanced_accuracy_score(y_raw, classes[(oof / prior**b).argmax(1)])
        for b in BETA_GRID
    )


def cb_space(trial: optuna.Trial) -> dict:
    return {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 10),
        "random_strength": trial.suggest_float("random_strength", 1e-8, 10.0, log=True),
    }


def main():
    train = pd.read_pickle(DATASET_DIR / "train_features_slim.pkl")
    test = pd.read_pickle(DATASET_DIR / "test_features_slim.pkl")
    print(f"train: {train.shape}, test: {test.shape}")

    X, y_raw = train[FEATURES].copy(), train[TARGET_COL]
    X_test = test[FEATURES].copy()

    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y_raw), index=y_raw.index)
    classes = le.classes_
    prior = pd.Series(y_raw).value_counts().reindex(classes).to_numpy() / len(y_raw)

    cat_cols = [c for c in X.columns if str(X[c].dtype) in ("object", "category")]
    for c in cat_cols:
        X[c] = X[c].astype(str)
        X_test[c] = X_test[c].astype(str)
    print("cat_cols:", cat_cols)

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    fold_splits = list(cv.split(X, y))

    def objective(trial: optuna.Trial) -> float:
        from catboost import CatBoostClassifier, Pool
        params = cb_space(trial)
        params.update({
            "loss_function": "MultiClass",
            "iterations": 1000,
            "random_seed": SEED,
            "verbose": False,
            "task_type": "GPU",
            "devices": "0",
        })
        oof = np.zeros((len(X), N_CLASSES))
        for tr_idx, val_idx in fold_splits:
            X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
            model = CatBoostClassifier(**params)
            model.fit(
                Pool(X_tr, y_tr, cat_features=cat_cols),
                eval_set=Pool(X_val, y_val, cat_features=cat_cols),
                early_stopping_rounds=50, verbose=False,
            )
            oof[val_idx] = model.predict_proba(X_val)
        return calibrated_score(oof, y_raw, classes, prior)

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    t0 = time.time()
    study.optimize(objective, n_trials=200, timeout=TIME_BUDGET_SECONDS, show_progress_bar=True)
    elapsed = time.time() - t0
    print(f"\nOptuna finished: {len(study.trials)} trials in {elapsed/60:.1f} min")
    print(f"Best value: {study.best_value:.5f}")
    print(f"Best params: {study.best_params}")

    with open("/kaggle/working/best_params_cb_full.json", "w") as f:
        json.dump(study.best_params, f, indent=2)

    # 最良パラメータでOOF/testを再学習して保存
    from catboost import CatBoostClassifier, Pool
    best_params = study.best_params.copy()
    best_params.update({
        "loss_function": "MultiClass", "iterations": 1000, "random_seed": SEED,
        "verbose": False, "task_type": "GPU", "devices": "0",
    })
    oof_preds = np.zeros((len(X), N_CLASSES))
    test_preds = np.zeros((len(X_test), N_CLASSES))
    for tr_idx, val_idx in fold_splits:
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        model = CatBoostClassifier(**best_params)
        model.fit(
            Pool(X_tr, y_tr, cat_features=cat_cols),
            eval_set=Pool(X_val, y_val, cat_features=cat_cols),
            early_stopping_rounds=50, verbose=False,
        )
        oof_preds[val_idx] = model.predict_proba(X_val)
        test_preds += model.predict_proba(X_test) / N_SPLITS

    final_score = calibrated_score(oof_preds, y_raw, classes, prior)
    print(f"Final calibrated OOF (re-trained best params) = {final_score:.5f}")

    np.save("/kaggle/working/oof_165_cb_full.npy", oof_preds)
    np.save("/kaggle/working/test_165_cb_full.npy", test_preds)
    print("saved: best_params_cb_full.json, oof_165_cb_full.npy, test_165_cb_full.npy")


if __name__ == "__main__":
    main()
